In [0]:
# Databricks notebook source
# ============================================================================
# Proyecto Medallion - Catalogo "cobranzas"
# Paso 2: Bronze (cuentas, promesas) -> Silver (cuentas)
# ============================================================================

from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.sql.types import DecimalType, IntegerType, StringType
 
CATALOG = "cobranzas"
BRONZE_CUENTAS = f"{CATALOG}.bronze.cuentas"
BRONZE_PROMESAS = f"{CATALOG}.bronze.promesas"
SILVER_CUENTAS = f"{CATALOG}.silver.cuentas"
SILVER_RECHAZADOS = f"{CATALOG}.silver.cuentas_rechazados"
 
# ---------------------------------------------------------------------------
# 1. Bronze -> Silver column mapping  (tal como lo diste)
# ---------------------------------------------------------------------------
BRONZE_SILVER_MAPPING = {
    "Jefatura": "jefatura",
    "Responsable": "responsable",
    "Gestor": "gestor",
    "Num_Prestamo_o_VP": "numero_de_prestamo_vp",
    "Cliente_Unico": "numero_cliente_unico",
    "Numero_DNI": "dni",
    "NOMBRE": "nombre",
    "Saldo_L": "saldo_l",
    "Saldo_USD": "saldo_usd",
    "Saldo_Total_USD": "saldo_total_usd",
    "Saldo_Tot_Honorarios": "saldo_total_honorarios",
    "Fecha_Asignacion": "fecha_asignacion",
    "Estado_Cartera": "estado_cartera",
    "Producto": "producto",
    "Segmentacion_Prod": "segmentacion_producto",
    "Tipo_Producto": "tipo_producto",
    "Fecha_Castigo": "fecha_castigo",
    "Fecha_Ult_Pago_Lps": "fecha_ultimo_pago",
    "Base_Laboral": "base_laboral",
    "Tipo_Empresa": "tipo_empresa",
    "Lugar_Trabajo": "lugar_trabajo",
    "Bienes_Inmuebles": "bienes_inmuebles",
    "Correo_Electronico": "email",
    "Cambio_Telefonos": "telefonos",
    "Fecha_Ult_Visita": "fecha_ultima_visita",
    "Celular": "celular",
    "Gestion_Historica": "gestion_historica",
    "Gestion_Diaria": "gestion_diaria",
    "Fecha_Ult_Gestion": "fecha_ultima_gestion",
    "Caracterizacion_del_mes": "caracterizacion_del_mes",
    "Tabla_Codigos_Gestion": "codigos_gestion",
    "Sub_Caracterizacion": "sub_caracterizacion",
    "Monto_Promesa": "monto_promesa",
    "Fecha_Promesa": "fecha_promesa",
    "Fecha_Solicitud_Docto": "fecha_solicitud_documento",
    "Fecha_Recibo_Docto": "fecha_recepcion_documentos",
    "Fecha_Present_Demanda": "fecha_presentacion_demanda",
    "Cuantia_Demanda": "cuantia_demandada_lps",
    "Cuantia_Demanda_USD": "cuantia_demandada_usd",
    "Tipo_Demanda": "tipo_demanda",
    "Estado_Demanda": "estado_demanda",
    "Caracterizacion_Judicial": "caracterizacion_judicial",
    "Fecha_Ult_Mov_Judicial": "fecha_ultimo_movimiento_judicial",
    "Gestion_Judicial_Historica": "gestion_judicial_historica",
    "Fecha_Revision_Expediente": "fecha_ultima_revision_expediente",
    "Fecha_Requerimiento": "fecha_requerimiento",
    "Fecha_Embargo": "fecha_embargo",
    "Embargo_de": "embargo_de",
    "Cantidad_Retenida_Mens": "cantidad_mensual_retenida",
    "Juzgado": "juzgado",
    "Expediente": "expediente",
    "Juez": "juez",
    "Ciudad_presen_demanda": "ciudad_demanda",
    "Ciudad": "ciudad",
    "Monto_Mensual_L": "monto_mensual_lps_promesa",
    "Monto_Mensual_$": "monto_mensual_dolar_promesa",
    "Fecha_de_pago": "fecha_de_pago_promesa",
    "Tipo_Arreglo": "tipo_arreglo_promesa",
    "Meses": "meses_promesa",
    "Fecha_Inicio": "fecha_inicio_promesa",
    "Fecha_Final": "fecha_final_promesa",
    "Observacion": "estado_promesa",
}
 
DATE_FORMATS = ["M/d/yyyy", "MM/dd/yyyy", "d/M/yyyy", "dd/MM/yyyy"]
 
CUENTAS_BRONZE_COLS = [
    "Jefatura", "Responsable", "Gestor", "Num_Prestamo_o_VP", "Cliente_Unico",
    "Numero_DNI", "NOMBRE", "Saldo_L", "Saldo_USD", "Saldo_Total_USD",
    "Saldo_Tot_Honorarios", "Fecha_Asignacion", "Estado_Cartera", "Producto",
    "Segmentacion_Prod", "Tipo_Producto", "Fecha_Castigo", "Fecha_Ult_Pago_Lps",
    "Base_Laboral", "Tipo_Empresa", "Lugar_Trabajo", "Bienes_Inmuebles",
    "Correo_Electronico", "Cambio_Telefonos", "Fecha_Ult_Visita", "Celular",
    "Gestion_Historica", "Gestion_Diaria", "Fecha_Ult_Gestion",
    "Caracterizacion_del_mes", "Tabla_Codigos_Gestion", "Sub_Caracterizacion",
    "Monto_Promesa", "Fecha_Promesa", "Fecha_Solicitud_Docto", "Fecha_Recibo_Docto",
    "Fecha_Present_Demanda", "Cuantia_Demanda", "Cuantia_Demanda_USD",
    "Tipo_Demanda", "Estado_Demanda", "Caracterizacion_Judicial",
    "Fecha_Ult_Mov_Judicial", "Gestion_Judicial_Historica",
    "Fecha_Revision_Expediente", "Fecha_Requerimiento", "Fecha_Embargo",
    "Embargo_de", "Cantidad_Retenida_Mens", "Juzgado", "Expediente", "Juez",
    "Ciudad_presen_demanda", "Ciudad",
]
 
PROMESAS_BRONZE_COLS = [
    "Cuenta", "Monto_Mensual_L", "Monto_Mensual_$", "Fecha_de_pago", "Tipo_Arreglo",
    "Meses", "Fecha_Inicio", "Fecha_Final", "Observacion",
]
 
# ---------------------------------------------------------------------------
# 2. Configuration per column: type, normalization, validations
# --------------------------------------------------------------------------

COLUMN_CONFIG = {
    "jefatura":                 {"type": "string", "normalize": {}, "mandatory": True},
    "responsable":              {"type": "string", "normalize": {}},
    "gestor":                   {"type": "string", "normalize": {}, "mandatory": True},
    "numero_de_prestamo_vp":    {"type": "string", "mandatory": True, "case": "none"},
    "numero_cliente_unico":     {"type": "string", "mandatory": True, "case": "none"},
    "dni":                      {"type": "string", "mandatory": True, "case": "none"},
    "nombre":                   {"type": "string", "mandatory": True},
    "saldo_l":                  {"type": "decimal", "minimum": 0},
    "saldo_usd":                {"type": "decimal", "minimum": 0},
    "saldo_total_usd":          {"type": "decimal", "minimum": 0},
    "saldo_total_honorarios":   {"type": "decimal", "minimum": 0},
    "fecha_asignacion":         {"type": "date"},
    "estado_cartera": {
        "type": "string", "normalize": {},
        "allowed_values": ["Judicial", "Extrajudicial", "Activa"],
    },
    "producto": {
        "type": "string",
        "normalize": {
            "TC CASTIGO": ["tc castigo", "Tc Castigo", "TC Castigo"],
            "PTMO CASTIGO": ["ptmo castigo", "Ptmo Castigo", "PTMO CASTigo"],
            "PTMO ACTIVO": ["ptmo activo", "Ptmo Activo", "PTMO Activo"],
            "TC 210": ["tc 210", "Tc 210"],
        },
        "allowed_values": ["TC CASTIGO", "PTMO CASTIGO", "PTMO ACTIVO", "TC 210"],
    },
    "segmentacion_producto": {
        "type": "string",
        "normalize": {
            "Banco Cb": ["Ficohsa CB", "Ficohsa Cb", "ficohsa cb", "FICOHSA CB", "Banco CB", "banco cb"],
            "Ft": ["FT", "ft"],
            "Banca De Consumo": ["Banca de Consumo", "banca de consumo", "BANCA DE CONSUMO"],
            "Banca Hipotecaria": ["banca hipotecaria", "BANCA HIPOTECARIA"],
            "Banca Emprendedores": ["banca emprendedores", "BANCA EMPRENDEDORES"],
            "Banca Pyme": ["Banca PYME", "banca pyme", "BANCA PYME"],
            "Banca Emprendedores Pequeño": ["Banca Emprendedores Pequeno", "banca emprendedores pequeño"],
            "Banca Emprendedores Grande": ["banca emprendedores grande", "BANCA EMPRENDEDORES GRANDE"],
            "Cb": ["CB", "cb"],
            "Pequeñas Empresas": ["Pequenas Empresas", "pequeñas empresas", "PEQUEÑAS EMPRESAS"],
        },
    },
    "tipo_producto": {
        "type": "string", "normalize": {
            "BancoCredito": ["Ficocredito"],
            "Other Personal": ["Citi Personal"],
            "OtherAuto": ["Ficoauto"],
            "Pil Turbo": ["Pil Turbo", "Pil_Turbo"]
        },
    },
    "fecha_castigo":              {"type": "date"},
    "fecha_ultimo_pago":          {"type": "date"},
    "base_laboral": {
        "type": "string", "normalize": {}, "allowed_values": ["Si", "No"],
    },
    "tipo_empresa": {
        "type": "string",
        "normalize": {
            "Privada": ["Privado", "Priivado", "privada", "PRIVADA"],
            "Publica": ["Publico", "publica", "PUBLICA"],
        },
        "allowed_values": [
            "Educacion Publica", "Comerciante Individual", "Privada",
            "Publica", "Educacion Privada", "Maquila",
        ],
    },
    "lugar_trabajo":              {"type": "string"},
    "bienes_inmuebles": {
        "type": "string",
        "normalize": {
            "Si": ["SI", "si"],
            "No": ["NO", "no"],
            "Si Aval": ["SI AVAL", "si aval"],
            "Pendiente": ["PENDIENTE", "pendiente"],
            "Si Rep Legal": ["SI REP LEGAL", "si rep legal"],
        },
        "allowed_values": ["Si", "No", "Si Aval", "Pendiente", "Si Rep Legal"],
    },
    "email":                      {"type": "string", "regex_format": r"^[^@\s]+@[^@\s]+\.[^@\s]+$", "case": "none"},
    "telefonos":                  {"type": "string", "case": "none"},
    "fecha_ultima_visita":        {"type": "date"},
    "celular":                    {"type": "string", "case": "none"},
    "gestion_historica":          {"type": "string"},
    "gestion_diaria":             {"type": "string"},
    "fecha_ultima_gestion":       {"type": "date"},
    "caracterizacion_del_mes": {
        "type": "string",
        "normalize": {
            "Acuerdo Parcial": ["ACUERDO PARCIAL", "acuerdo parcial"],
            "Acuerdo Total": ["ACUERDO TOTAL", "acuerdo total"],
            "Cancelado Con Descuento": ["CANCELADO CON DESCUENTO"],
            "Fuera del País": ["CLIENTE FUERA DEL PAÍS"],
            "Demandado": ["DEMANDADO", "demandado"],
            "Demandado/Acp": ["DEMANDADO/ACP"],
            "Demandado/Pro": ["DEMANDADO/PRO"],
            "Demandado/Act": ["DEMANDADO/ACT"],
            "Demandado/Cfp": ["DEMANDADO/CFP"],
            "Demandado/Csp": ["DEMANDADO/CSP"],
            "Demandado/Embargado": ["DEMANDADO/EMBARGADO", "demandado/embargado", "Demandado/Embargado"],
            "Demandado/ILO SB ST": ["DEMANDADO/ILO SB ST"],
            "Sin documento para demanda": ["SIN DOCUMENTO PARA DEMANDA"],
            "Demandado/Inembargable": ["DEMANDADO/INEMBARGABLE", "demandado/inembargable", "Demandado/Inembargable"],
            "Demandado/Ldd": ["DEMANDADO/LDD"],
            "Demandado/LOC SB ST": ["DEMANDADO/LOC SB ST"],
            "Documentos": ["DOCUMENTOS"],
            "Documentos/CSP": ["DOCUMENTOS/CSP"],
        },
    },
    "codigos_gestion": {
        "type": "string", "normalize": {}, "case": "upper",
        "allowed_values": [
            "ACP", "ACT", "CADES", "CFP", "DISBT", "DLSBT", "DMACP", "DMACT", "DMAD",
            "DMCFP", "DMCSP", "DMD", "DMEMB", "DMINB", "DMLDD", "DMPRO", "DOACP", "DOCSP",
            "DOACT", "DOCAD", "DOLDD", "DOPRO", "DOCTO", "ILCT", "ILST", "INV",
            "LDD", "LOSBT", "LOST", "PRO", "SALBA", "SDPDM", "PDMD"
        ],
    },
    "sub_caracterizacion": {
        "type": "string",
        "normalize": {
            "Data Demográfica Retroalimentada": [
                "Datademografica Retroalimentado",
                "Datra Demografica Retroalimentado",
                "Datademografica Retroalimentado",
                "Data Retroalimentado",
                "Data Demografica",
                "Data  Demográfica",
                "Data Demografica Retroalimentada",
                "Data Demografica Retroalimentado",
                "Data Demograrfica Retroalimentado",
                "Data Demográfica Retroalimenado",
            ],
            "Data Demográfica Retroalimentada, Ilocalizado": [
                "Data Demográfica Retroalimentado Ilocalizado",
            ],
        },
    },
    "monto_promesa":              {"type": "decimal", "minimum": 0},
    "fecha_promesa":              {"type": "date"},
    "fecha_solicitud_documento":  {"type": "date"},
    "fecha_recepcion_documentos": {"type": "date"},
    "fecha_presentacion_demanda": {"type": "date"},
    "cuantia_demandada_lps":      {"type": "decimal", "minimum": 0},
    "cuantia_demandada_usd":      {"type": "decimal", "minimum": 0},
    "tipo_demanda": {
        "type": "string",
        "normalize": {
            "Ejecución de Título Extrajudicial": [
                "EJECUCIÓN DE TÍTULO EXTRAJUDICIAL",
                "ejecucion de titulo extrajudicial",
                "Ejecucion de Titulo Extrajudicial",
            ],
            "Ejecución Directa y Exclusiva Sobre un Bien Inmueble Hipotecado": [
                "EJECUCIÓN DIRECTA Y EXCLUSIVA SOBRE UN BIEN INMUEBLE HIPOTECADO",
                "ejecucion directa y exclusiva sobre un bien inmueble hipotecado",
            ],
            "Demanda de Ejecución Directa y Exclusiva Sobre un Bien Pignorado": [
                "DEMANDA DE EJECUCIÓN DIRECTA Y EXCLUSIVA SOBRE UN BIEN PIGNORADO",
                "demanda de ejecucion directa y exclusiva sobre un bien pignorado",
            ],
        },
    },
    "estado_demanda": {
        "type": "string", "normalize": {}, "allowed_values": ["Activa", "Inactiva"],
    },
    "caracterizacion_judicial": {
        "type": "string",
        "normalize": {
            "Admitida": ["ADMITIDA", "admitida"],
            "Embargado": ["EMBARGADO", "embargado"],
            "Demanda Presentada": ["DEMANDA PRESENTADA", "demanda presentada"],
            "Para Embargo": ["PARA EMBARGO", "para embargo"],
            "Para Requerimiento": ["PARA REQUERIMIENTO", "para requerimiento"],
            "Prescrita": ["PRESCRITA"],
            "Deuda Cancelada": ["DEUDA CANCELADA"],
            "Notificacion de Costas": ["NOTIFICACION DE COSTAS"],
            "Oficio / Solicitud de Informacion de Sueldo": ["OFICIO/SOLICITUD DE INFORMACION DE SUELDO"],
            "Subsanación": ["SUBSANACION"],
            "Nulidad de Actuaciones": ["NULIDAD DE ACTUACIONES"],
            "Oficios/investigacion de bienes muebles e inmuebles": ["OFICIOS/INVESTIGACION DE BIENES MUEBLES E INMUEBLES"],
            "Exhorto": ["EXHORTO"],
            "Cantidades retenidas": ["CANTIDADES RETENIDAS"],
            "Publicacion de subasta": ["PUBLICACION DE SUBASTA"],
            "Inembargable": ["INEMBARGABLE"],
            "Preclusion": ["PRECLUSION"],
            "Ya fue requerido": ["YA FUE REQUERIDO"],
            "Publicacion de edictos": ["PUBLICACION DE EDICTOS"],
            "Inadmitida/devolucion de doc": ["INADMITIDA/DEVOLUCION DE DOC"],
            "Tasacion de costas": ["TASACION DE COSTAS"],
            "Null": ["NULL"],
            "Archivo general": ["ARCHIVO GENERAL"],
            "Demanda extraviada": ["DEMANDA EXTRAVIADA"],
            "Subasta de inmueble": ["SUBASTA DE INMUEBLE"],
            "Oficios/investigacion de domicilio": ["OFICIOS/INVESTIGACION DE DOMICILIO"],
            "Solicitud de subasta": ["SOLICITUD DE SUBASTA"],
        },
    },
    "fecha_ultimo_movimiento_judicial": {"type": "date"},
    "gestion_judicial_historica":       {"type": "string"},
    "fecha_ultima_revision_expediente": {"type": "date"},
    "fecha_requerimiento":              {"type": "date"},
    "fecha_embargo":                    {"type": "date"},
    "embargo_de": {
        "type": "string",
        "normalize": {
            "Vehiculo": ["Vehículo", "vehiculo", "VEHICULO"],
            "Cuenta bancaria": ["Cuenta Bancaria", "cuenta bancaria", "CUENTA BANCARIA"],
            "Propiedad": ["propiedad", "PROPIEDAD"],
            "Salario": ["salario", "SALARIO"],
        },
        "allowed_values": ["Cuenta bancaria", "Propiedad", "Salario", "Vehiculo"],
    },
    "cantidad_mensual_retenida":  {"type": "decimal", "minimum": 0},
    "juzgado":                    {"type": "string"},
    "expediente":                 {"type": "string", "case": "none"},
    "juez":                       {"type": "string"},
    "ciudad_demanda": {
        "type": "string",
        "normalize": {
            "San Pedro Sula": ["SAN PEDRO SULA", "san pedro sula"],
            "Choloma": ["CHOLOMA", "choloma"],
            "La Ceiba": ["LA CEIBA", "la ceiba"],
            "Puerto Cortés": ["PUERTO CORTÉS", "Puerto Cortes", "puerto cortes", "PUERTO CORTES"],
            "Villanueva": ["VILLANUEVA", "villanueva"],
            "El Progreso": ["EL PROGRESO", "el progreso"],
            "Tela": ["TELA", "tela"],
            "Tegucigalpa": ["TEGUCIGALPA", "tegucigalpa"],
        },
    },
    "ciudad": {
        "type": "string",
        "normalize": {
            "San Pedro Sula": ["SAN PEDRO SULA", "san pedro sula", "San Pedro Sua"],
            "Choloma": ["CHOLOMA", "choloma"],
            "La Ceiba": ["LA CEIBA", "la ceiba"],
            "Puerto Cortés": ["PUERTO CORTÉS", "Puerto Cortes", "puerto cortes"],
            "Villanueva": ["VILLANUEVA", "villanueva"],
            "El Progreso": ["EL PROGRESO", "el progreso"],
            "Tela": ["TELA", "tela"],
            "Santa Cruz de Yojoa": ["Santa Cruz De Yojoa", "SANTA CRUZ DE YOJOA"],
        },
    },
    "monto_mensual_lps_promesa":   {"type": "decimal", "minimum": 0},
    "monto_mensual_dolar_promesa": {"type": "decimal", "minimum": 0},
    "fecha_de_pago_promesa":       {"type": "date"},
    "tipo_arreglo_promesa": {
        "type": "string", "case": "upper",
        "allowed_values": ["PRO", "ACP", "ACT", "LDD"],
    },
    "meses_promesa": {"type": "integer", "minimum": 1},
    "fecha_inicio_promesa":  {"type": "date"},
    "fecha_final_promesa":   {"type": "date"},
    "estado_promesa": {
        "type": "string",
        "normalize": {
            "Postergada": ["Postergó"]},
        "allowed_values": ["Pendiente", "Cumplida", "Incumplida", "Postergada"],
    },
}
 
# ---------------------------------------------------------------------------
# 3. Motor generico: normalizar, castear y validar segun COLUMN_CONFIG
# --------------------------------------------------------------------------
 
def apply_case_expr(col: "F.Column", case_type: str) -> "F.Column":
    """Aplica el tipo de mayus/minus por defecto cuando el valor no vino en el
    diccionario 'normalize' de la columna."""
    if case_type == "upper":
        return F.upper(col)
    if case_type == "lower":
        return F.lower(col)
    if case_type == "none":
        return col
    # "title" (default): primera letra de cada palabra en mayuscula, resto minuscula
    return F.initcap(F.lower(col))
 
 
def normalize_expr(col: "F.Column", normalize_map: dict, case_type: str = "title") -> "F.Column":
    """Mapea variantes -> valor canonico usando un lookup map. Si no hay match,
    aplica 'case_type' como fallback en vez de dejar el valor crudo (que es lo
    que causaba que llegaran valores en MAYUSCULA/minuscula libre a silver)."""
    fallback = apply_case_expr(col, case_type)
    if not normalize_map:
        return fallback
    flat = {}
    for canonical, variants in normalize_map.items():
        flat[canonical] = canonical
        for v in variants:
            flat[v] = canonical
    pairs = []
    for k, v in flat.items():
        pairs.extend([F.lit(k), F.lit(v)])
    lookup = F.create_map(*pairs)
    return F.coalesce(lookup[col], fallback)
 
 
def parse_date_expr(col: "F.Column") -> "F.Column":
    """Intenta parsear con cada formato de DATE_FORMATS, en orden, hasta que uno funcione."""
    candidates = [F.to_date(F.try_to_timestamp(col, F.lit(fmt))) for fmt in DATE_FORMATS]
    return F.coalesce(*candidates)
 
 
def parse_decimal_expr(col: "F.Column", precision=18, scale=2) -> "F.Column":
    cleaned = F.regexp_replace(F.trim(col), ",", "")
    is_numeric = cleaned.rlike(r"^-?\d+(\.\d+)?$")
    return F.when(is_numeric, cleaned.cast(DecimalType(precision, scale))).otherwise(
        F.lit(None).cast(DecimalType(precision, scale))
    )
 
 
def parse_integer_expr(col: "F.Column") -> "F.Column":
    cleaned = F.trim(col)
    is_int = cleaned.rlike(r"^-?\d+$")
    return F.when(is_int, cleaned.cast(IntegerType())).otherwise(F.lit(None).cast(IntegerType()))
 
 
def apply_column_config(df: DataFrame, column_config: dict) -> DataFrame:
    """
    Para cada columna presente en column_config:
      1. normaliza (si es string y trae 'normalize'; si no matchea, aplica 'case')
      2. castea segun 'type' (string/decimal/integer/date)
      3. acumula errores de validacion (mandatory/minimum/allowed_values/regex_format)
    Agrega _dq_errors (array<string>) y _is_valid (boolean).
    """
    error_cols = []
 
    for col_name, cfg in column_config.items():
        if col_name not in df.columns:
            continue
 
        raw = F.col(col_name)
        col_type = cfg.get("type", "string")
 
        if col_type == "string":
            working = normalize_expr(raw, cfg.get("normalize", {}), cfg.get("case", "title"))
            working = F.trim(working)
            working = F.when(working == "", None).otherwise(working)
        elif col_type == "decimal":
            working = parse_decimal_expr(raw, 18, 2)
        elif col_type == "integer":
            working = parse_integer_expr(raw)
        elif col_type == "date":
            working = parse_date_expr(raw)
        else:
            working = raw
 
        df = df.withColumn(col_name, working)
        current = F.col(col_name)
 
        col_errors = []
 
        if cfg.get("mandatory"):
            col_errors.append(
                F.when(current.isNull(), F.lit(f"{col_name}: obligatorio y vacio/no parseable"))
            )
 
        if "minimum" in cfg:
            minimum = cfg["minimum"]
            col_errors.append(
                F.when(
                    current.isNotNull() & (current < F.lit(minimum)),
                    F.lit(f"{col_name}: menor al minimo permitido ({minimum})"),
                )
            )
 
        if "allowed_values" in cfg:
            allowed = cfg["allowed_values"]
            col_errors.append(
                F.when(
                    current.isNotNull() & (~current.isin(allowed)),
                    F.concat(F.lit(f"{col_name}: valor fuera de catalogo -> '"), current, F.lit("'")),
                )
            )
 
        if "regex_format" in cfg:
            pattern = cfg["regex_format"]
            col_errors.append(
                F.when(
                    current.isNotNull() & (~current.rlike(pattern)),
                    F.concat(F.lit(f"{col_name}: no cumple formato -> '"), current, F.lit("'")),
                )
            )
 
        error_cols.extend(col_errors)
 
    if error_cols:
        df = df.withColumn("_dq_errors", F.array_compact(F.array(*error_cols)))
    else:
        df = df.withColumn("_dq_errors", F.array().cast("array<string>"))
 
    df = df.withColumn("_is_valid", F.size(F.col("_dq_errors")) == 0)
    return df
 
# ---------------------------------------------------------------------------
# 4. Leer bronze y quedarnos con la foto mas reciente por llave de negocio
# --------------------------------------------------------------------------
 
bronze_cuentas = spark.table(BRONZE_CUENTAS)
bronze_promesas = spark.table(BRONZE_PROMESAS)
 
# 4.1 cuentas: 1 fila por numero_de_prestamo_vp = la mas reciente por ingestion_timestamp
w_cuenta = (
    bronze_cuentas
    .withColumn(
        "_rn",
        F.row_number().over(
            __import__("pyspark").sql.Window
            .partitionBy("Num_Prestamo_o_VP")
            .orderBy(F.col("ingestion_timestamp").desc())
        ),
    )
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)
 
# 4.2 promesas: 1 fila por (Cuenta, Fecha_de_pago, Observacion) = la mas reciente
w_promesa = (
    bronze_promesas
    .withColumn(
        "_rn",
        F.row_number().over(
            __import__("pyspark").sql.Window
            .partitionBy("Cuenta", "Fecha_de_pago", "Observacion")
            .orderBy(F.col("ingestion_timestamp").desc())
        ),
    )
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)
 
cuentas_sel = w_cuenta.select(
    *CUENTAS_BRONZE_COLS,
    F.col("source_file").alias("bronze_source_file_cuenta"),
)
promesas_sel = w_promesa.select(
    *PROMESAS_BRONZE_COLS,
    F.col("source_file").alias("bronze_source_file_promesa"),
)
 
 
# ---------------------------------------------------------------------------
# 5. LEFT JOIN cuentas -> promesas por numero_de_prestamo_vp = Cuenta
# --------------------------------------------------------------------------
joined = cuentas_sel.join(
    promesas_sel,
    cuentas_sel["Num_Prestamo_o_VP"] == promesas_sel["Cuenta"],
    "left",
)
 
 
# ---------------------------------------------------------------------------
# 6. Renombrar bronze -> silver segun BRONZE_SILVER_MAPPING
# ---------------------------------------------------------------------------
select_exprs = []
for bronze_col, silver_col in BRONZE_SILVER_MAPPING.items():
    select_exprs.append(F.col(bronze_col).alias(silver_col))
 
renamed = joined.select(
    *select_exprs,
    "bronze_source_file_cuenta",
    "bronze_source_file_promesa",
)
 
# ---------------------------------------------------------------------------
# 7. Aplicar normalizacion + casteo + validaciones (COLUMN_CONFIG)
# --------------------------------------------------------------------------
validated = apply_column_config(renamed, COLUMN_CONFIG)
 
#-------------------------------------------------------------------------
# 8. Llave subrogada (silver_key) para el MERGE
#    numero_de_prestamo_vp + llave natural de la promesa (o 'NA' si no tiene)
#    Llave natural de la promesa = fecha_de_pago_promesa + estado_promesa.
 
validated = validated.withColumn(
    "silver_key",
    F.sha2(
        F.concat_ws(
            "||",
            F.coalesce(F.col("numero_de_prestamo_vp"), F.lit("NA")),
            F.coalesce(F.col("fecha_de_pago_promesa").cast("string"), F.lit("NA")),
            F.coalesce(F.col("estado_promesa"), F.lit("NA")),
        ),
        256,
    ),
).withColumn("silver_processed_at", F.current_timestamp())
 
 
#-------------------------------------------------------------------------
# 9. Separar validos (-> silver.cuentas via MERGE) de invalidos (-> cuarentena)
#--------------------------------------------------------------------------
silver_cols = [
    "silver_key",
    "jefatura", "responsable", "gestor", "numero_de_prestamo_vp", "numero_cliente_unico",
    "dni", "nombre", "saldo_l", "saldo_usd", "saldo_total_usd", "saldo_total_honorarios",
    "fecha_asignacion", "estado_cartera", "producto", "segmentacion_producto", "tipo_producto",
    "fecha_castigo", "fecha_ultimo_pago", "base_laboral", "tipo_empresa", "lugar_trabajo",
    "bienes_inmuebles", "email", "telefonos", "fecha_ultima_visita", "celular",
    "gestion_historica", "gestion_diaria", "fecha_ultima_gestion", "caracterizacion_del_mes",
    "codigos_gestion", "sub_caracterizacion", "monto_promesa", "fecha_promesa",
    "fecha_solicitud_documento", "fecha_recepcion_documentos", "fecha_presentacion_demanda",
    "cuantia_demandada_lps", "cuantia_demandada_usd", "tipo_demanda", "estado_demanda",
    "caracterizacion_judicial", "fecha_ultimo_movimiento_judicial", "gestion_judicial_historica",
    "fecha_ultima_revision_expediente", "fecha_requerimiento", "fecha_embargo", "embargo_de",
    "cantidad_mensual_retenida", "juzgado", "expediente", "juez", "ciudad_demanda", "ciudad",
    "monto_mensual_lps_promesa", "monto_mensual_dolar_promesa", "fecha_de_pago_promesa",
    "tipo_arreglo_promesa", "meses_promesa", "fecha_inicio_promesa", "fecha_final_promesa",
    "estado_promesa",
    "bronze_source_file_cuenta", "bronze_source_file_promesa", "silver_processed_at",
]
 
validos = validated.filter(F.col("_is_valid")).select(*silver_cols)
rechazados = validated.filter(~F.col("_is_valid")).select(
    *[c for c in silver_cols if c != "silver_key"],
    "silver_key",
    "_dq_errors",
)
 
#--------------------------------------------------------------------------
# 10. MERGE de validos hacia silver.cuentas (upsert por silver_key)
#-------------------------------------------------------------------------

validos.createOrReplaceTempView("stg_silver_cuentas")
 
set_clause = ", ".join([f"t.{c} = s.{c}" for c in silver_cols if c != "silver_key"])
insert_cols = ", ".join(silver_cols)
insert_vals = ", ".join([f"s.{c}" for c in silver_cols])
 
spark.sql(f"""
    MERGE INTO {SILVER_CUENTAS} AS t
    USING stg_silver_cuentas AS s
    ON t.silver_key = s.silver_key
    WHEN MATCHED THEN UPDATE SET {set_clause}
    WHEN NOT MATCHED THEN INSERT ({insert_cols}) VALUES ({insert_vals})
""")
 
#-------------------------------------------------------------------------
# 11. Cuarentena: se sobrescribe completa en cada corrida
# ------------------------------------------------------------------------

rechazados_str = rechazados.select(
    *[F.col(c).cast("string").alias(c) if c not in ("silver_key", "_dq_errors", "silver_processed_at")
      else F.col(c)
      for c in rechazados.columns]
)
 
(
    rechazados_str.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_RECHAZADOS)
)
 
print(f"Validos cargados a {SILVER_CUENTAS}: {validos.count()}")
print(f"Rechazados en {SILVER_RECHAZADOS}: {rechazados.count()}")
 